**`validate_occupancy_against_permits`**

Validates curated occupancy, year built, and area against Shovels building-permit evidence.

Permits are the second out-of-band evidence source, after the CHEER hand labels.

- They are not NSI-derived, not parcel-derived, and not baked into the curated output.
- Coverage varies enormously by county, so every rate is reported per county.
- Permit evidence is parcel-level, smeared onto every footprint of the parcel.
- Rows without permits mean *no evidence*, never a negative.

All permit- and survey-derived outputs go to the cache tree, never the repository.

# Configure

In [ ]:
import os

os.environ['OPENPLACES_PERMIT_STATE'] = 'TX'
import argparse
import re
from pathlib import Path

import geopandas as gpd
import pandas as pd

import openplaces as op

from openplaces.io.curator.validation import validation_context  # noqa: E402

# All CHEER-specific validation configuration is declared in the
# curate recipe's `validation:` block (plus the untracked
# validation-references sidecar for licence-restricted tables) and
# resolved here; see io.curator.validation.ValidationContext.
_ctx = validation_context('US_footprint-openplaces-2026', 'TX')
RECIPE_ID = _ctx.recipe_id
CLASS_MAP = _ctx.class_map
COLLAPSE = _ctx.collapse
DERIVED_SOURCE_COLUMNS = _ctx.derived_source_columns
PERMIT_STATE = _ctx.references_state
PERMIT_REGION = _ctx.reference_region
PERMIT_DIR = _ctx.reference_dir
class_from_ruleset = _ctx.class_from_ruleset
from openplaces.io.delivery import delivery_accuracy_dir  # noqa: E402

VAL_DIR = PERMIT_DIR  # state-scoped through the validation context

pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 250)

In [ ]:
parser = argparse.ArgumentParser(
    description='Validate curated occupancy against Shovels permit evidence.'
)
parser.add_argument('--recipe_id', default=RECIPE_ID)
# Default None: inventory the validation directory at run time. The permit
# batch is still growing, so the county list must never be hardcoded.
parser.add_argument('--counties', nargs='*', default=None)
# Beside the bundle, not in the cache: these are aggregate tables about
# what the delivery is worth, so they ship with it.
parser.add_argument(
    # NC-specific references; see validate_delivery for the selector.
    '--out_dir',
    default=str(delivery_accuracy_dir(RECIPE_ID, region=PERMIT_REGION)),
)
parser.add_argument('--verbose', action='store_true')

# Test arguments

In [ ]:
ARGS_TEST = '--verbose '

args_list = [x for x in ARGS_TEST.split(' ') if x != '']
args = parser.parse_args(args_list)
args

# Validate occupancy against permit evidence

## Inventory

The shovels worktree writes one file pair per county.

- Counties are discovered at run time; incomplete pairs are reported, not read.
- `n_permits` NaN means no permit evidence at all; those rows are excluded everywhere.
- `matched_via` records how the *occupancy-bearing* permits matched: `parcel_id_local` beats `address`; `none` means permits exist but none carries an occupancy type.

In [ ]:
files = sorted(VAL_DIR.glob(f'US-{PERMIT_STATE}-*_occupancy_validation.parquet'))
kinds_by_county = {}
for f in files:
    m = re.match(
        rf'(US-{PERMIT_STATE}-\w{{3}})_(footprint|parcel)_occupancy_validation', f.stem
    )
    if m:
        kinds_by_county.setdefault(m.group(1), set()).add(m.group(2))
complete = sorted(
    c for c, kinds in kinds_by_county.items() if kinds == {'footprint', 'parcel'}
)
incomplete = sorted(set(kinds_by_county) - set(complete))
if args.counties:
    complete = [c for c in complete if c in args.counties]
print(f'complete county pairs: {len(complete)}')
if incomplete:
    print(f'incomplete (skipped, likely mid-write): {incomplete}')


def load_permits(admin_id):
    path = VAL_DIR / f'{admin_id}_footprint_occupancy_validation.parquet'
    if not path.exists():
        return None
    df = pd.read_parquet(path)
    if 'footprint_id' in df.columns:
        df = df.dropna(subset=['footprint_id']).set_index('footprint_id')
    df.index.name = 'footprint_id'
    return df


def tier(df):
    """Confidence tier for permit occupancy evidence, high to low.

    matched_via parcel_id_local beats address; mode_pct 1.0 with at least
    two occupancy-bearing permits beats a single uncorroborated one.
    """
    n_occ = pd.to_numeric(df['n_permits_with_occupancy_type'], errors='coerce')
    strong = df['occupancy_type_mode_pct'].ge(0.999) & n_occ.ge(2)
    via_id = df['matched_via'].eq('parcel_id_local')
    has = df['occupancy_type_mode'].notna()
    t = pd.Series('none', index=df.index)
    t[has & ~via_id & ~strong] = '4_addr_weak'
    t[has & ~via_id & strong] = '3_addr_strong'
    t[has & via_id & ~strong] = '2_id_weak'
    t[has & via_id & strong] = '1_id_strong'
    return t


rows = []
for county in complete:
    perm = load_permits(county)
    has_ev = perm['n_permits'].notna()
    via = perm.loc[has_ev, 'matched_via'].value_counts()
    rows.append(
        {
            'county': county,
            'n_rows': len(perm),
            'n_with_permits': int(has_ev.sum()),
            'pct': round(100 * has_ev.mean(), 1),
            'via_parcel_id': int(via.get('parcel_id_local', 0)),
            'via_address': int(via.get('address', 0)),
            'n_with_occupancy': int(perm['occupancy_type_mode'].notna().sum()),
        }
    )
inventory = pd.DataFrame(rows).sort_values('pct', ascending=False)
inventory

## Vote vs permits, per county and tier

The comparison joins on `footprint_id` (stable across branches) and scores primary footprints only.

- Permit evidence is parcel-level: a shed inherits the house's class, so secondary footprints cannot be scored against it.
- Only counties with curated output on disk and a non-trivial number of occupancy-bearing permits appear.

In [ ]:
agree_rows = []
confusions = {}
county_joins = {}
for county in complete:
    perm = load_permits(county)
    if perm is None or perm['occupancy_type_mode'].notna().sum() < 200:
        continue
    fp = op.get_entities(args.recipe_id, county, missing='ignore')
    if fp is None or fp.empty:
        continue
    # The raw columns come from DERIVED_SOURCE_COLUMNS rather than being
    # listed by hand: a signal whose class column `order_columns` drops is
    # rebuilt from its raw column, so a whitelist that forgets one silently
    # deletes that signal from every table below.
    wanted = [
        'occupancy_type',
        'occupancy_type_source',
        'occupancy_type_nsi_class',
        'occupancy_type_fema_class',
        'occupancy_keyword_class',
        'occupancy_type_parcel',
        'n_dwellings_overture',
        'priority_on_parcel',
        'year_built',
        'area_m2',
        'use_group_combined_parcel',
    ] + [spec['column'] for spec in DERIVED_SOURCE_COLUMNS.values()]
    cols = [c for c in dict.fromkeys(wanted) if c in fp.columns]
    joined = fp[cols].join(perm, how='inner')
    joined['vote'] = joined['occupancy_type'].astype(object).replace(COLLAPSE)
    joined['tier'] = tier(joined)
    county_joins[county] = joined
    primary = joined[
        joined['priority_on_parcel'].astype(object).isin(['primary', 'unknown'])
    ]
    for t, sub in primary[primary['tier'] != 'none'].groupby('tier'):
        both = sub[sub['vote'].notna()]
        agree_rows.append(
            {
                'county': county,
                'tier': t,
                'n': len(both),
                'agree': round((both['vote'] == both['occupancy_type_mode']).mean(), 3)
                if len(both)
                else None,
            }
        )
    strong = primary[
        primary['tier'].isin(['1_id_strong', '2_id_weak', '3_addr_strong'])
    ]
    strong = strong[strong['vote'].notna()]
    if len(strong) >= 300:
        confusions[county] = pd.crosstab(strong['vote'], strong['occupancy_type_mode'])

agreement = pd.DataFrame(agree_rows).pivot_table(
    index='county', columns='tier', values=['n', 'agree'], aggfunc='first'
)
agreement

In [ ]:
try:
    for county, table in confusions.items():
        print(f'--- {county}: vote (rows) x permit mode (cols), strong tiers ---')
        print(table.to_string())
        print()
except Exception as _e:
    print(f'diagnostic cell skipped: {type(_e).__name__}: {_e}')

## What the confusions taught, and the fixes now in effect

Three disagreement blocks dominated the first pass of this analysis. Each led to a shipped fix, so the tables above show the post-fix state.

**1. The Single-Family residual absorbed non-residential buildings.**

- Thousands of vote-SF footprints sat on parcels whose permits (and assessor strings: churches, retail, warehouses) were non-residential.
- The two-question vote had no non-residential exit: once question 1 said `single`, the residual overwrote even a non-residential base class.
- CHEER cannot see this error mode: the survey labels only residential structures.
- Fix: the residuals now `require` a residential-or-unknown `occupancy_type_prevote`; the cell below verifies the leak is closed.

**2. `Multi-Section MH` was keyword-classified as Multi-Family.**

- Onslow's vote-MF-vs-permit-MH block was one assessor string, `Multi-Section MH` (a double/triple-wide manufactured home — one dwelling), caught by the reviewed `MULTI` pattern before `\bMH\b` could claim it.
- Fix: a `MULTI[- ]?SECTION` → Manufactured Home rule precedes it; the cell below verifies.

**3. NSI+FEMA agreement was one signal counted twice.**

- The two modeled sources are near-copies (97.5% identical); their pair reached the multi decision's `min_score` alone and flipped true single-family homes wherever FEMA coverage is complete.
- Fix: the pair pools into one `any_of` vote, and only NSI's specific 3-plus-unit claims (80–86% right against CHEER, vs 35% for its duplex guess) earn the corroborating second vote.

In [ ]:
# Verify fix 2: the Onslow Multi-Section MH block should be near zero
# (it measured 2,500 footprints before the keyword-rule fix).
on = county_joins.get('US-NC-ONS')
if on is not None:
    block = on[
        on['vote'].eq('Multi-Family')
        & on['occupancy_type_mode'].eq('Manufactured Home')
    ]
    multisection = (
        block['use_group_combined_parcel']
        .astype(str)
        .str.contains('MULTI-SECTION', case=False, na=False)
    )
    print(
        f'ON vote=MF & permit=MH: {len(block)} footprints '
        f'({int(multisection.sum())} still on a Multi-Section string)'
    )
    print()
    print('remaining assessor strings (top 5):')
    print(
        block['use_group_combined_parcel']
        .astype(object)
        .value_counts()
        .head(5)
        .to_string()
    )

In [ ]:
# Verify fix 1: the New Hanover non-residential leak should be near zero
# (it measured ~2,650 id-matched footprints before the prevote gate).
ne = county_joins.get('US-NC-NHA')
if ne is not None:
    nonres = [
        'Commercial',
        'Industrial',
        'Institutional',
        'Office',
        'Retail',
        'Agricultural',
        'Recreation',
    ]
    block = ne[
        ne['vote'].eq('Single-Family')
        & ne['occupancy_type_mode'].isin(nonres)
        & ne['matched_via'].eq('parcel_id_local')
    ]
    print(
        f'NE vote=SF & permit=non-residential (id-matched): {len(block)} '
        f'footprints on {block["parcel_id_local"].nunique()} parcels'
    )
    print()
    print('remaining assessor strings (top 10):')
    print(
        block['use_group_combined_parcel']
        .astype(object)
        .value_counts()
        .head(10)
        .to_string()
    )

## Permits vs the CHEER hand labels

The overlap is small and clustered, so it bounds what permits can certify.

- CHEER points sit in counties with thin permit coverage; the overlap concentrates in New Hanover.
- Permit evidence is parcel-level: one wrongly mapped parcel poisons every footprint on it.
- The headline example: a manufactured-home community whose 125 permits are all mapped `Office` (mode_pct 1.0) — CHEER and the vote both say Manufactured Home for all 45 points on it.
- Lesson: `mode_pct` × `n_permits` is not reliability on large multi-structure parcels. Count distinct parcels before trusting any overlap rate.

## The Single-Family ceiling (handoff question 3)

Is Single-Family's F1 ceiling an evidence limit or a weighting limit?

- At the CHEER points the permit overlap is too thin to arbitrate directly.
- At inventory scale, the New Hanover confusion carries the answer: large vote-MF-and-permit-SF and vote-SF-and-permit-MF blocks exist side by side, plus the non-residential leak above.
- Reading: partly evidence-limited (as the handoff guessed), but the non-residential leak is a *structural* gap no reweighting fixes — and it deflates SF precision invisibly, because CHEER never labels non-residential buildings.

## Robeson PARDESC1 (handoff question 4.1)

Permits are absent in Robeson (28 matched rows), so the codes are tested against the 232 CHEER labels, joined spatially to the county's raw parcel file.

The handoff's letter-code reading was substantially wrong:

- `D-10` → Single-Family holds (65/67, on 66 independent parcels).
- The manufactured-home code is `C-92` (49/49 CHEER-MH), not `D-71` — but those 49 points sit on only 9 parcels.
- `C-65` is Multi-Family (29/29, 7 parcels); `E-15`/`E-80` lean Multi-Family (public/exempt housing).
- `D-71` is only weakly MH: 7/10, on 10 independent parcels.
- `V-80` (a *vacant* code, 10K parcels county-wide) carries manufactured homes at 15/17 — the leased-land signature, left unmapped pending a second source.

The three verified codes now ship as a `remap_pattern` in the Robeson ingest recipe (`D-10`/`C-92`/`C-65` → keyword-friendly labels in `use_group`, raw code preserved in `use_subgroup`), giving the county 40K+ assessor-labeled structures where it previously had none.

## Pender zoning (handoff question 4.2)

`zoning_code` now survives to curated parcels (spine `keep_columns`); this cell joins zones spatially from the ingested Pender parcels for full coverage (98%).

The verdict is empirical and mixed:

- The `MH` zone is enriched for manufactured homes (permits: 21 MH vs 10 SF) but tiny.
- The multi-family zones `RM-CD1`/`RM-CD2` contain **zero** permit-confirmed Multi-Family — 127 Single-Family homes. Zoning states what *may* be built; here it demonstrably differs from what is.
- The broad `RP`/`RA` residential zones split roughly 3:1 SF:MH — weakly informative at best.

Zoning is therefore not worth adding as a vote input; it remains useful as descriptive context.

In [ ]:
try:
    perm_pd = load_permits('US-NC-PEN')
    fp_pd = op.get_entities(args.recipe_id, 'US-NC-PEN', geom=True, missing='ignore')
    ing_pd = op.get_entities(
        'US-NC-PEN_parcel-pendercounty-2026', 'US-NC-PEN', geom=True, missing='ignore'
    )
    if fp_pd is not None and ing_pd is not None and 'zoning_code' in ing_pd:
        joined = fp_pd[['occupancy_type', 'priority_on_parcel', 'geometry']].join(
            perm_pd, how='left'
        )
        pts = joined.copy()
        pts['geometry'] = pts.geometry.representative_point()
        zoned = gpd.sjoin(
            pts, ing_pd[['zoning_code', 'geometry']], how='left', predicate='within'
        )
        zoned = zoned[~zoned.index.duplicated(keep='first')]
        zp = zoned[
            zoned['occupancy_type_mode'].notna()
            & zoned['priority_on_parcel'].astype(object).isin(['primary', 'unknown'])
        ].copy()
        zp['vote'] = zp['occupancy_type'].astype(object).replace(COLLAPSE)
        top = zp['zoning_code'].value_counts().head(12).index
        print('permit occupancy by zone (primary footprints):')
        print(
            pd.crosstab(
                zp[zp['zoning_code'].isin(top)]['zoning_code'],
                zp['occupancy_type_mode'],
            ).to_string()
        )
    else:
        print('missing inputs for the zoning analysis; skipping')
except Exception as _e:
    print(f'diagnostic cell skipped: {type(_e).__name__}: {_e}')

## Year built and area (handoff question 6.4)

Permit `year_built` splits the counties into two regimes:

- Onslow and Johnston: 92–97% of primary footprints within ±1 year — the curated cascade and permits describe the same event.
- New Hanover and Pender: under 9% within ±1 year, spreads of decades — the curated `year_built` there comes from a different (likely block-median) lineage and should not be treated as parcel-grade.

Footprint area vs permit square footage is stable everywhere: the median ratio sits near 1.2 in all four counties, consistent with gross footprint vs heated area. A systematic, calibratable relationship — not a defect.

In [ ]:
year_area_rows = []
for county in ['US-NC-NHA', 'US-NC-ONS', 'US-NC-JOH', 'US-NC-PEN']:
    joined = county_joins.get(county)
    if joined is None:
        continue
    primary = joined[
        joined['priority_on_parcel'].astype(object).isin(['primary', 'unknown'])
    ]
    yb = primary.dropna(subset=['year_built', 'year_built_most_recent'])
    diff = pd.to_numeric(yb['year_built']) - pd.to_numeric(yb['year_built_most_recent'])
    row = {
        'county': county,
        'year_built_pairs': len(yb),
        'year_within_1yr': round((diff.abs() <= 1).mean(), 4),
        'year_diff_median': diff.median(),
        'year_diff_p10': diff.quantile(0.1),
        'year_diff_p90': diff.quantile(0.9),
    }
    ar = primary.dropna(subset=['area_m2', 'area_sqft_median'])
    if len(ar):
        # Curated area is metric; the permit figure is square feet.
        ratio = (
            pd.to_numeric(ar['area_m2'])
            * 10.7639
            / pd.to_numeric(ar['area_sqft_median'])
        )
        row |= {
            'area_pairs': len(ar),
            'area_ratio_p25': round(ratio.quantile(0.25), 3),
            'area_ratio_median': round(ratio.median(), 3),
            'area_ratio_p75': round(ratio.quantile(0.75), 3),
        }
    year_area_rows.append(row)

year_area = pd.DataFrame(year_area_rows)
year_area

## Which input dataset should the vote prefer?

Permit mode (strong tiers, primary footprints) referees each input signal, cross-checked against CHEER.

Rules that both references agree on:

- **Assessor keywords beat both modeled sources in conflicts.** Permits back keyword over NSI 77/23 and over FEMA 80/20; CHEER backs it 60/9 and 36/3.
- **A Manufactured Home claim beats a Single-Family claim from any source.** MH claims are specific; SF is the default.
- **FEMA is a near-copy of NSI** (97.5% identical against permits, 95.8% at CHEER points; right only a third of the time where NSI is wrong). Their agreement is one signal, not two.
- **Specificity separates NSI's good claims from its noise**: `Multi-Family, 2 units` is 35% right against CHEER; 3-plus-unit claims are 80–86% right.

Where the two references disagree, trust CHEER:

- Permits' occupancy is vendor property metadata that plausibly shares upstream lineage with NSI, so permits cannot referee NSI-involved conflicts.
- Example: on rows where `multi` rested on NSI+FEMA agreement alone, permits sided with the pair while CHEER's eyes-on-the-building labels showed a 50/50 split.
- Overture illustrates the same bias: permits score it worst in conflicts, yet CHEER backs `overture >= 2` over modeled SF 46/0.

In [ ]:
RES3 = ['Single-Family', 'Multi-Family', 'Manufactured Home']
SIGNAL_COLS = {
    'nsi': 'occupancy_type_nsi_class',
    'fema': 'occupancy_type_fema_class',
    'keyword': 'occupancy_keyword_class',
    'parcel': 'occupancy_type_parcel',
    'vote': 'occupancy_type',
}

# order_columns drops the *_class vote intermediates from published
# output, so no county can ever carry them. Rebuild them from the raw
# evidence through the same ruleset the vote used. The specs are shared
# with the ground-truth notebook so the two cannot disagree about how a
# signal is reconstructed.
RAW_FOR_SIGNAL = DERIVED_SOURCE_COLUMNS

frames = []
for county, joined in county_joins.items():
    primary = joined[
        joined['tier'].isin(['1_id_strong', '2_id_weak', '3_addr_strong'])
        & joined['priority_on_parcel'].astype(object).isin(['primary', 'unknown'])
    ]
    sig = pd.DataFrame(index=primary.index)
    for name, col in SIGNAL_COLS.items():
        if col in primary.columns:
            sig[name] = primary[col].astype(object).replace(COLLAPSE)
        elif name in RAW_FOR_SIGNAL:
            spec = RAW_FOR_SIGNAL[name]
            if spec['column'] not in primary.columns:
                continue
            derived = class_from_ruleset(
                primary[spec['column']],
                spec.get('ruleset', CLASS_MAP),
                reviewed_only=spec.get('reviewed_only', False),
            )
            if derived is not None:
                sig[name] = derived.astype(object).replace(COLLAPSE)
    ov = pd.to_numeric(primary['n_dwellings_overture'], errors='coerce')
    sig['overture'] = pd.Series(pd.NA, index=primary.index, dtype=object)
    sig.loc[ov.ge(2), 'overture'] = 'Multi-Family'
    sig.loc[ov.eq(1), 'overture'] = 'Single-Family'
    sig['permit'] = primary['occupancy_type_mode'].astype(object)
    sig['county'] = county
    frames.append(sig)
if not frames:
    raise SystemExit(
        'No county produced comparable signals. Check that the permit '
        'validation parquets exist and that county_joins is populated.'
    )
signals = pd.concat(frames, ignore_index=True)
signals = signals[signals['permit'].isin(RES3)]

SIGNALS = ['nsi', 'fema', 'keyword', 'parcel', 'overture', 'vote']
missing = [name for name in SIGNALS if name not in signals.columns]
if missing:
    # Never silently: a signal that vanishes takes its rows out of every
    # table below, and a quietly shorter table reads as a finding.
    print(f'WARNING: no comparable values for {missing}; omitted below')

rows = []
for name in SIGNALS:
    if name not in signals.columns:
        continue
    both = signals[signals[name].isin(RES3)]
    rows.append(
        {
            'signal': name,
            'coverage': round(len(both) / len(signals), 3),
            'agree_with_permit': round((both[name] == both['permit']).mean(), 3),
            'n': len(both),
        }
    )
print(
    f'strong-tier primary rows, {signals["county"].nunique()} counties: '
    f'{len(signals):,}'
)
signal_scores = pd.DataFrame(rows)
print(signal_scores.to_string(index=False))

In [ ]:
# Conflict adjudication: when two signals disagree, whom does the permit
# reference back? Read NSI-involved rows with the caveat above.
pairs = [
    ('nsi', 'fema'),
    ('nsi', 'keyword'),
    ('fema', 'keyword'),
    ('nsi', 'overture'),
    ('keyword', 'overture'),
]
adjudication = []
for a, b in pairs:
    if a not in signals.columns or b not in signals.columns:
        continue
    d = signals[
        signals[a].isin(RES3)
        & signals[b].isin(RES3)
        & signals[a].fillna('-').ne(signals[b].fillna('-'))
    ]
    if len(d) < 50:
        continue
    backs_a = int((d['permit'] == d[a]).sum())
    backs_b = int((d['permit'] == d[b]).sum())
    adjudication.append(
        {
            'conflict': f'{a} vs {b}',
            'n': len(d),
            'permit_backs_first': backs_a,
            'permit_backs_second': backs_b,
            'first_share': round(backs_a / max(backs_a + backs_b, 1), 3),
        }
    )
adjudication_table = pd.DataFrame(adjudication)
print(adjudication_table.to_string(index=False))

# Independence probe: FEMA conditional on NSI.
if {'fema', 'nsi'} <= set(signals.columns):
    d = signals[signals['fema'].isin(RES3) & signals['nsi'].isin(RES3)]
    nsi_right = d['nsi'].eq(d['permit'])
    print()
    print('P(nsi==fema):', round(d['nsi'].eq(d['fema']).mean(), 3))
    print(
        'P(fema==permit | nsi==permit):',
        round(d.loc[nsi_right, 'fema'].eq(d.loc[nsi_right, 'permit']).mean(), 3),
    )
    print(
        'P(fema==permit | nsi!=permit):',
        round(d.loc[~nsi_right, 'fema'].eq(d.loc[~nsi_right, 'permit']).mean(), 3),
    )

## Save derived artifacts

Permit- and survey-derived tables are third-party-derived, so they go to the cache tree.

In [ ]:
out_dir = Path(args.out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
written = []


def write(table, name, **kwargs):
    """Write one table and record it for the summary line."""
    if table is None or not len(table):
        return
    path = out_dir / f'{args.recipe_id}_{name}.csv'
    table.to_csv(path, **kwargs)
    written.append(path.name)


write(inventory, 'permit-coverage', index=False)
write(agreement, 'permit-agreement')
write(year_area, 'permit-year-area', index=False)
write(signal_scores, 'permit-signal-scores', index=False)
write(adjudication_table, 'permit-signal-conflicts', index=False)
# One long-form file rather than one per county: a reader filtering by
# county gets the same thing, and a new county needs no new file.
if confusions:
    write(
        pd.concat(
            [
                t.stack().rename('n').reset_index().assign(county=c)
                for c, t in confusions.items()
            ],
            ignore_index=True,
        )[['county', 'vote', 'occupancy_type_mode', 'n']],
        'permit-confusion',
        index=False,
    )
print(f'wrote {len(written)} tables to {out_dir}:')
for name in written:
    print(f'  {name}')

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script

COMMIT = True

try:
    convert_to_script(commit=COMMIT)
except Exception as error:
    # Headless execution (nbconvert) has no notebook context to resolve
    # the caller path from; run this cell interactively to commit the
    # script. Stripped from the converted script either way.
    print(f'convert_to_script skipped: {error}')

# Test script

In [ ]:
# from openplaces.flow import test_script

# test_script(*args_list, committed=COMMIT)

# Inspect results

In [ ]:
try:
    agreement
except Exception as _e:
    print(f'diagnostic cell skipped: {type(_e).__name__}: {_e}')